# Load TSpec Data

In [1]:
import os
import re
from bs4 import BeautifulSoup
from markdown import markdown

from app.config import get_logger, TSPEC_DATA, CHUNKS_FILE, APP_ENV
from app.utils.chunking import save_chunks

Using Device: cuda


/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_chatbotUI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logger = get_logger(__name__)

In [3]:
print(f"APP_ENV: {APP_ENV}")
print(f"TSPEC_DATA: {TSPEC_DATA}")
print(f"CHUNKS_FILE: {CHUNKS_FILE}")

APP_ENV: test_file
TSPEC_DATA: ../../../../../Dataset/TSpec-LLM/3GPP-clean/Rel-18/28_series/28532-i00.md
CHUNKS_FILE: ../../../files/chunks/tspec_chunks_test_rel_18_28_28532.pkl


In [4]:
# # This function loads markdown files, preserves formatting, and converts HTML tables to readable markdown/plaintext.
# def load_tspec_data(directory):
#     """
#     Load .md files while preserving formatting (text and tables) in a readable form.
#     For each .md file (filtered to Rel-18 / 28_series like the previous loader),
#     returns a dict with:
#       - release, series, spec (filename without .md)
#       - original_md: original markdown text
#       - processed_text: extracted text with tables converted to markdown/plaintext
#       - html: HTML generated from the markdown (useful for debug/visualization)
#     """
#     data = []

#     for release in os.listdir(directory):
#         release_path = os.path.join(directory, release)
#         if os.path.isdir(release_path) and release == "Rel-18":
#             for series in os.listdir(release_path):
#                 series_path = os.path.join(release_path, series)
#                 if os.path.isdir(series_path) and series == "28_series":
#                     for file in os.listdir(series_path):
#                         if file.endswith('.md'):
#                             file_path = os.path.join(series_path, file)
#                             with open(file_path, 'r', encoding='utf-8') as f:
#                                 content = f.read()
#                                 spec_name = file[:-3]

#                                 # Used to be a filter. Use a filter here if want to test with a specific file, e.g., if release == "Rel-18" and series == "28_series" and spec_name[:5] == "28532": # Old filter
                                
#                                 # Convert markdown to HTML (support tables and fenced code)
#                                 html = markdown(content, extensions=["tables", "fenced_code", "codehilite"])

#                                 # Use BeautifulSoup to find HTML tables and convert them to readable markdown/plaintext
#                                 soup = BeautifulSoup(html, "html.parser")

#                                 # Extract text from the resulting HTML, preserving line breaks
#                                 processed_text = soup.get_text(separator="\n", strip=True)

#                                 # Normalize multiple blank lines
#                                 processed_text = re.sub(r"\n\s*\n+", "\n\n", processed_text).strip()

#                                 data.append({
#                                     "release": release,
#                                     "series": series,
#                                     "spec": spec_name,
#                                     "original_md": content,
#                                     "processed_text": processed_text,
#                                     "html": str(soup)
#                                 })
#     return data

In [5]:
# This function loads markdown files, preserves formatting, and converts HTML tables to readable markdown/plaintext.
def load_tspec_data(tspec_path: str = None):
    """
    Load .md files while preserving formatting (text and tables) in a readable form.
    
    Fully generic behavior based ONLY on the type of path received from TSPEC_DATA:
      - If path is a single .md FILE     → loads only that file (test_file mode)
      - If path is a DIRECTORY           → recursively walks ALL subdirectories and loads every .md file
    
    Metadata extraction (release / series) is 100% dynamic based on actual folder structure at runtime.
    No hardcoded folder names (Rel-18, 28_series, etc.).
    
    Returns list of dicts with:
      - release, series, spec (filename without .md)
      - original_md: original markdown text
      - processed_text: extracted text with tables converted to markdown/plaintext
      - html: HTML generated from the markdown (useful for debug/visualization)
    """
    if tspec_path is None:
        tspec_path = TSPEC_DATA

    data = []
    logger.info(f"Starting load from: {tspec_path}  (APP_ENV={os.getenv('APP_ENV', 'prod')})")

    # ===============================================
    # 1. SINGLE FILE MODE (test_file)
    # ===============================================
    if os.path.isfile(tspec_path):
        logger.info(f"Detected single file: {os.path.basename(tspec_path)}")
        files_to_process = [(os.path.dirname(tspec_path), os.path.basename(tspec_path))]

    # ===============================================
    # 2. DIRECTORY MODE (test or prod) – fully generic using os.walk
    # ===============================================
    else:
        logger.info("Detected directory – using os.walk() for complete agnostic traversal")
        files_to_process = []
        for root, _, files in os.walk(tspec_path):
            for file in files:
                if file.endswith('.md'):
                    files_to_process.append((root, file))

    # ===============================================
    # Process all collected files (single or many)
    # ===============================================
    for root, file in files_to_process:
        file_path = os.path.join(root, file)
        logger.debug(f"Processing file: {file}")

        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        spec_name = file[:-3]

        # ===========================================
        # GENERIC extraction of release and series
        # ===========================================
        # series  = immediate parent folder of the .md file
        # release = folder one level above the series folder
        series_dir = os.path.basename(root)
        release_dir = os.path.basename(os.path.dirname(root)) if os.path.dirname(root) else ""

        release = release_dir if release_dir else "unknown"
        series  = series_dir if series_dir else "unknown"

        # Convert markdown to HTML (support tables and fenced code)
        html = markdown(content, extensions=["tables", "fenced_code", "codehilite"])

        # Use BeautifulSoup to extract clean text (tables become readable plaintext)
        soup = BeautifulSoup(html, "html.parser")

        # Extract text from the resulting HTML, preserving line breaks
        processed_text = soup.get_text(separator="\n", strip=True)

        # Normalize multiple blank lines
        processed_text = re.sub(r"\n\s*\n+", "\n\n", processed_text).strip()

        data.append({
            "release": release,
            "series": series,
            "spec": spec_name,
            "original_md": content,
            "processed_text": processed_text,
            "html": str(soup)
        })

    logger.info(f"Finished loading → {len(data)} documents")
    return data

In [6]:
# In Settings change to TSPEC_DATA=TSPEC_DATA_TEST_FILE to use as TEST dataset of 1 file or change APP_ENV to "test" and use TSPEC_DATA_TEST_FILE in the config
# Use this when need test that is the same as use the extraction with this filter -> # if release == "Rel-18" and series == "28_series" and spec_name[:5] == "28532": # Old filter

tspec_data = load_tspec_data(TSPEC_DATA)

2026-02-22 18:24:41 [INFO] __main__: Starting load from: ../../../../../Dataset/TSpec-LLM/3GPP-clean/Rel-18/28_series/28532-i00.md  (APP_ENV=test_file)
2026-02-22 18:24:41 [INFO] __main__: Detected single file: 28532-i00.md
2026-02-22 18:24:41 [DEBUG] __main__: Processing file: 28532-i00.md
2026-02-22 18:24:41 [DEBUG] MARKDOWN: Successfully loaded extension "markdown.extensions.tables.TableExtension".
2026-02-22 18:24:41 [DEBUG] MARKDOWN: Successfully loaded extension "markdown.extensions.fenced_code.FencedCodeExtension".
2026-02-22 18:24:41 [DEBUG] MARKDOWN: Successfully loaded extension "markdown.extensions.codehilite.CodeHiliteExtension".
2026-02-22 18:24:41 [INFO] __main__: Finished loading → 1 documents


In [7]:
tspec_data

[{'release': 'Rel-18',
  'series': '28_series',
  'spec': '28532-i00',
  'original_md': '+-----------------------------------------------------------------+---+\n| 3GPP TS 28.532 V18.0.0 (2023-09)                                |   |\n+=================================================================+===+\n| Technical Specification                                         |   |\n+-----------------------------------------------------------------+---+\n| 3rd Generation Partnership Project;                             |   |\n|                                                                 |   |\n| Technical Specification Group Services and System Aspects;      |   |\n|                                                                 |   |\n| Management and orchestration;                                   |   |\n|                                                                 |   |\n| Generic management services                                     |   |\n|                                  

In [8]:
print(f"Sample document:\n{tspec_data[0]['original_md'][:10000]}")

Sample document:
+-----------------------------------------------------------------+---+
| 3GPP TS 28.532 V18.0.0 (2023-09)                                |   |
+=================================================================+===+
| Technical Specification                                         |   |
+-----------------------------------------------------------------+---+
| 3rd Generation Partnership Project;                             |   |
|                                                                 |   |
| Technical Specification Group Services and System Aspects;      |   |
|                                                                 |   |
| Management and orchestration;                                   |   |
|                                                                 |   |
| Generic management services                                     |   |
|                                                                 |   |
| (Release 18)                                 

In [9]:
# print(f"Total documents loaded: {len(tspec_data)}")
# print(f"Sample document: {tspec_data[0]['processed_text']}")
print(f"Sample document: {tspec_data[0]['html']}")


Sample document: <p>+-----------------------------------------------------------------+---+
| 3GPP TS 28.532 V18.0.0 (2023-09)                                |   |
+=================================================================+===+
| Technical Specification                                         |   |
+-----------------------------------------------------------------+---+
| 3rd Generation Partnership Project;                             |   |
|                                                                 |   |
| Technical Specification Group Services and System Aspects;      |   |
|                                                                 |   |
| Management and orchestration;                                   |   |
|                                                                 |   |
| Generic management services                                     |   |
|                                                                 |   |
| (Release 18)                              

# Build chunks (isolate Tables)

## Separate Text and Tables

In [10]:
from typing import List, Dict, Any
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [11]:
def is_table_line(line: str) -> bool:
    """
    Heuristic to detect if a line belongs to a 3GPP-style Markdown/ASCII table.
    """
    stripped = line.strip()
    if not stripped:
        return False
    # Classic separator: +---+---+ or similar
    if re.match(r'^\+[-+=| ]+\+$', stripped):
        return True
    # Row with | separators, starting or ending with |
    if '|' in stripped and (stripped.startswith('|') or stripped.endswith('|')):
        return True
    return False

def extract_tables_pure_md(md_content: str, release: str = "unknown", series: str = "unknown", spec: str = "unknown") -> tuple[str, List[Dict[str, str]]]:
    """
    Extract complete table blocks from raw Markdown text.
    Inserts a numbered placeholder in clean_text where each table was removed, with metadata.
    Returns:
        - clean_text: markdown without tables, but with placeholders like "[Extracted Table: 1 from spec (Release Y, Series Z)]"
        - tables: list of dicts with {'number': '1', 'content': full table markdown} (for easy numbering in chunks)
    """
    lines = md_content.splitlines(keepends=False)
    clean_lines: List[str] = []
    tables: List[Dict[str, str]] = []
    current_table: List[str] = []
    table_counter = 1  # Simple counter for table numbers (per document)
    i = 0

    while i < len(lines):
        line = lines[i].rstrip("\r\n")

        if is_table_line(line):
            current_table.append(line)
            i += 1
            # Continue collecting while it looks like table
            while i < len(lines) and is_table_line(lines[i].rstrip("\r\n")):
                current_table.append(lines[i].rstrip("\r\n"))
                i += 1

            # Save only if it looks like a real table (min 3 lines)
            if len(current_table) >= 3:
                table_md = "\n".join(current_table)
                tables.append({
                    'number': str(table_counter),
                    'content': table_md
                })

                # Insert placeholder in clean text (with number and metadata)
                placeholder = f"[Extracted Table: {table_counter} from {spec} (Release {release}, Series {series})]"
                clean_lines.append(placeholder)
                table_counter += 1
            else:
                # False positive → return to clean text
                clean_lines.extend(current_table)
            current_table = []
        else:
            clean_lines.append(line)
            i += 1

    clean_text = "\n".join(clean_lines)
    # Normalize excessive newlines
    clean_text = re.sub(r'\n{3,}', '\n\n', clean_text).strip()

    return clean_text, tables

In [12]:
# Run the extraction function
clean_text, tables = extract_tables_pure_md(tspec_data[0]['original_md'])

In [13]:
# Print results for debug
print("=== Clean Text (without tables) ===\n")
print(clean_text[:10000] + "..." if len(clean_text) > 500 else clean_text)  # Show first 500 chars

=== Clean Text (without tables) ===

[Extracted Table: 1 from unknown (Release unknown, Series unknown)]

[Extracted Table: 2 from unknown (Release unknown, Series unknown)]

 Contents {#contents .TT}

Foreword 14

1 Scope 16

2 References 16

3 Definitions and abbreviations 18

3.1 Definitions 18

3.2 Abbreviations 18

4 Overview 18

5 Void 18

6 Void 18

7 Void 18

8 Void 18

9 Void 18

10 Void 19

11 Management services -- Stage 2 19

11.1 Generic provisioning management service 19

11.1.0 Introduction 19

11.1.1 Operations and notifications 19

11.1.1.1 createMOI operation 19

11.1.1.1.1 Description 19

11.1.1.1.2 Input parameters 19

11.1.1.1.3 Output parameters 20

11.1.1.1.4 Results 20

11.1.1.2 getMOIAttributes operation 21

11.1.1.2.1 Definition 21

11.1.1.2.2 Input Parameters 21

11.1.1.2.3 Output Parameters 22

11.1.1.2.4 Results 23

11.1.1.3 modifyMOIAttributes operation 23

11.1.1.3.1 Description 23

11.1.1.3.2 Input parameters 24

11.1.1.3.3 Output parameters 26

11.1.1.3

In [14]:
print("\n=== Extracted Tables ===\n")
if tables:
    for table_dict in tables:
        print(f"Table Number: {table_dict['number']}")
        print(f"Content:\n{table_dict['content']}\n")
else:
    print("No tables found in sample.")


=== Extracted Tables ===

Table Number: 1
Content:
+-----------------------------------------------------------------+---+
| 3GPP TS 28.532 V18.0.0 (2023-09)                                |   |
+=================================================================+===+
| Technical Specification                                         |   |
+-----------------------------------------------------------------+---+
| 3rd Generation Partnership Project;                             |   |
|                                                                 |   |
| Technical Specification Group Services and System Aspects;      |   |
|                                                                 |   |
| Management and orchestration;                                   |   |
|                                                                 |   |
| Generic management services                                     |   |
|                                                                 |   |
| (Release 1

## Split Text and create chunks

In [15]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
    (r"\n\d+(\.\d+)*\s", "Section"),  # Matches 1 Scope, 4.3.1 Any, etc.
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # Keep header inside chunk for context
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,       # Good for 3GPP: fits most sections/tables
    chunk_overlap=300,     # Overlap for cross-section references
    separators=[
        "\n\n\n",
        r"\n\d+(\.\d+)*\s+[A-Za-z]",  # Prioritize numbered sections like "4.3.1 Definition"
        "\n# ", "\n## ", "\n### ",
        "\n\n", "\n", " ", ""
    ],
)

In [16]:
def divide_into_chunks(tspec_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Process loaded data into chunks.
    - Extracts tables as intact chunks first (with numbered placeholders in clean_text for coherence)
    - Splits remaining text with markdown_splitter + recursive_splitter
    - Adds metadata for RAG (release, series, spec, chunk_index, is_table, headers)
    """
    dataset_chunks = []
    chunk_index = 0

    for document in tspec_data:
        release = document['release']
        series = document['series']
        spec = document['spec']
        original_md = document['original_md']  # Use raw MD for accurate table extraction

        # Extract tables with placeholders and numbering (pass metadata)
        clean_text, table_blocks = extract_tables_pure_md(original_md, release=release, series=series, spec=spec)

        # Split the clean text (now with numbered placeholders)
        header_chunks = markdown_splitter.split_text(clean_text)

        # Refine with recursive splitter
        for header_chunk in header_chunks:
            char_chunks = recursive_splitter.split_text(header_chunk.page_content)
            for chunk_text in char_chunks:
                cleaned_chunk = chunk_text.strip()
                if cleaned_chunk:  # Skip empty chunks
                    dataset_chunks.append({
                        'release': release,
                        'series': series,
                        'spec': spec,
                        'content': cleaned_chunk,
                        'chunk_index': chunk_index,
                        'is_table': False,
                        'headers': header_chunk.metadata  # e.g., {'Section': '4.3.1'}
                    })
                    chunk_index += 1

        # Add each extracted table as a separate chunk (with number in title)
        for table_dict in table_blocks:
            table_number = table_dict['number']
            table_content = table_dict['content']
            enriched_table = f"**Spec {spec}, Release {release}, Series {series}**\nTable {table_number}\n\n{table_content}"
            dataset_chunks.append({
                'release': release,
                'series': series,
                'spec': spec,
                'content': enriched_table,
                'chunk_index': chunk_index,
                'is_table': True,
                'headers': {'Table': f'Full extracted table {table_number}'}
            })
            chunk_index += 1

    print(f"Generated {len(dataset_chunks)} chunks ({sum(1 for c in dataset_chunks if c['is_table'])} tables)")
    return dataset_chunks

In [17]:
tspec_chunks = divide_into_chunks(tspec_data)

Generated 745 chunks (119 tables)


In [18]:
# Check the result
print(f"Total chunks created: {len(tspec_chunks)}")
print(f"Example chunk:\n {tspec_chunks[100]}")
print(f"Example table chunk:\n {tspec_chunks[700]}")

Total chunks created: 745
Example chunk:
 {'release': 'Rel-18', 'series': '28_series', 'spec': '28532-i00', 'content': 'correlatedNotifications   O   Set of CorrelatedNotification related to this AlarmInformation.\nstateChangeDefinition     O   AlarmInformation.stateChange\nmonitoredAttributes       O   AlarmInformation.monitoredAttributes\nproposedRepairActions     O   AlarmInformaton.proposedRepairActions\nadditionalText            O   AlarmInformation.additionalText\nadditionalInformation     O   AlarmInformation.additionalInformation\nrootCauseIndicator        O   alarmInformation.rootCauseIndicator\nchangedAlarmAttributes    O   LIST OF SEQUENCE \\<AttributeName, OldAttributeValue\\>             The changed alarm attributes (name/value pairs) (with old values).  \n###### 11.2.1.2.8.3 Input parameters for notifications related to security alarm  \nThe notifyChangedAlarmGeneral notification is defined by Table\n11.2.1.1.4.2a-1, if the alarmType is equal to \\"Integrity Violation\\",

In [19]:
chunks_path = CHUNKS_FILE
save_chunks(tspec_chunks, chunks_path)

Chunks saved to ../../../files/chunks/tspec_chunks_test_rel_18_28_28532.pkl


# Build chunks (without isolate tables)

In [20]:
# from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
# import re

# # Configure headers for splitting
# headers_to_split_on = [
#     ("#", "Header 1"),
#     ("##", "Header 2"),
#     ("###", "Header 3"),
#     ("####", "Header 4"),
#     (r"\n\d+(\.\d+)*\s", "Section"),  # Matches 1 Scope, 4.3.1 Any, etc.
# ]

# # Initialize the MarkdownHeaderTextSplitter
# markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

# # Configure the RecursiveCharacterTextSplitter
# chunk_size = 1200
# chunk_overlap = 300
# separators=[
#         "\n\n\n",  # Big blocks (ex: after Contents)
#         r"\n\d+(\.\d+)*\s+[A-Za-z]",  # Prioritize numbered sections like "4.3.1 Definition"
#         "\n# ", "\n## ", "\n### ",
#         "\n\n", "\n", " ", ""
#     ]
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=chunk_size,
#     chunk_overlap=chunk_overlap,
#     separators=separators
# )

# # Function to divide content into chunks
# def divide_into_chunks(tspec_data):
#     dataset_chunks = []

#     for document in tspec_data:
#         release = document['release']
#         series = document['series']
#         spec = document['spec']
#         content = document['original_md']
        
#         # Split by Markdown headers
#         header_chunks = markdown_splitter.split_text(content)
        
#         # Further split the chunks by characters
#         for header_chunk in header_chunks:
#             char_chunks = text_splitter.split_text(header_chunk.page_content)
#             for chunk in char_chunks:
#                 dataset_chunks.append({
#                     'release': release,
#                     'series': series,
#                     'spec': spec,
#                     'content': chunk
#                 })

#     return dataset_chunks

In [21]:
# tspec_chunks = divide_into_chunks(tspec_data)

In [22]:
# # Check the result
# print(f"Total chunks created: {len(tspec_chunks)}")
# print(f"Example chunk:\n {tspec_chunks[10]}")

In [23]:
# chunks_path = CHUNKS_FILE
# save_chunks(tspec_chunks, chunks_path)